### Code to generate the model fit and forecasts

In [ ]:

import numpy as np
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import sys
import scipy.stats as st

dt = 0.1
tstart = 0
tlim = 200
t = np.arange(tstart, tlim, 1)

d_cut = 160

ndiv = 1/dt

### keep it always starting at 0
tmoh = np.arange(0, tlim, dt)

N_city = 1

from pathlib import Path

ROOT = Path.cwd().resolve()
while not ((ROOT / "PHU_Data").exists() and (ROOT / "data").exists()):
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Could not find the sum_of_sigmoid repository root. Open this notebook from the repository root or one of its subdirectories.")
    ROOT = ROOT.parent

PHU_path = ROOT / "PHU_Data"
datapath = ROOT / "data"
figpath = ROOT / "figs" / "predictions" / "synth_check2"
figpath.mkdir(parents=True, exist_ok=True)

Data = np.zeros([365,4])

files = os.listdir(PHU_path)

target_file1 = f'{PHU_path}/30-Toronto.csv'
target_file2 = f'{PHU_path}/34-York.csv'
target_file3 = f'{PHU_path}/04-Durham.csv'
target_file4 = f'{PHU_path}/22-PeelRegion.csv'


target_file5 = f'{datapath}/synthetic_case2_data.dat'

Data[:,0] = np.genfromtxt(target_file1, delimiter=',')
Data[:,1] = np.genfromtxt(target_file2, delimiter=',')
Data[:,2] = np.genfromtxt(target_file3, delimiter=',')
Data[:,3] = np.genfromtxt(target_file4, delimiter=',')


population_by_phu = np.genfromtxt(f'{PHU_path}/population_by_phu.csv', delimiter=',')


#### CHANgE HERE ########
# x_data = np.genfromtxt(target_file5, delimiter=',')
x_data = Data[:,0]
t_data = np.arange(0,tlim)


# Preallocate compartments
S = np.zeros((len(tmoh),N_city))
E = np.zeros((len(tmoh),N_city))
I = np.zeros((len(tmoh),N_city))
R = np.zeros((len(tmoh),N_city))
D = np.zeros((len(tmoh),N_city))
N = np.zeros((len(tmoh),N_city))


# For noisy synthetic model output
I_model = np.zeros((len(t),N_city))
I_synthetic = np.zeros((len(t),N_city))

 
total = np.zeros((N_city))

 ####### CHANGE HERE ######################
total[0] = population_by_phu[29,1]
 #####################################

# Model parameters - Taken from Southern Ontario - COVID MBE paper
gamma_e = 1/15
gamma_i = 1/5
gamma_r = 1/11
gamma_d = 1/750

beta_e = np.zeros((len(tmoh),N_city)) 
beta_i = np.zeros((len(tmoh),N_city))

beta_calibrated = np.zeros((len(tmoh),N_city))

print("total population in PHU1",total[0])

#### PEEL - 234 , 1451022

#### Toronto - 430 , 2794356

### York - 212, 1173334

### Durham - 94   - 696992


In [ ]:


sample_path = datapath

true_path = datapath / "estimated_truth_all.csv"

beta_true_path = datapath / "beta_estimated_200.dat"

all_truth = np.genfromtxt(true_path, delimiter=',')

beta_truth = np.loadtxt(beta_true_path)

S_mean = np.zeros((len(tmoh),N_city))
E_mean = np.zeros((len(tmoh),N_city))
I_mean = np.zeros((len(tmoh),N_city))
R_mean = np.zeros((len(tmoh),N_city))
D_mean = np.zeros((len(tmoh),N_city))
beta_mean = np.zeros((len(tmoh),N_city))


samples = np.loadtxt(f'{sample_path}/muVec_real_MAP.dat')

# samples = np.loadtxt(f'{sample_path}/muVec_real_case2.dat')

print(samples.shape)

Nsamples = len(samples[0,:])

I_pdf = np.zeros((tlim-d_cut,Nsamples))


for ns in range(Nsamples):


### Toronto
    a0 = samples[0,ns]
    a1 = samples[1,ns]
    t1 =  20
    a2 = samples[2,ns]
    t2 =  35
    a3 =  samples[3,ns]
    t3 = 60
    a4 =  samples[4,ns]
    t4 = 80
    a5 =  samples[5,ns]
    t5 =  140

    a6 =  samples[6,ns]
    t6 = 180
    a7 =  samples[7,ns]
    t7 =  190


        # Initial Conditions
    ####### CHANGE HERE ######################
    E[0,0] = samples[8,ns]
    I[0,0] = samples[9,ns]
    #####################################
    R[0,0] = 0
    D[0,0] = 0
    N[0,0] = total[0]
    S[0,0] = N[0,0] - E[0,0] - I[0,0] - R[0,0] - D[0,0]



    beta_i[:,0] = a0  + a1/(1 + np.exp((t1-tmoh))) +  a2/(1 + np.exp((t2-tmoh)))  + a3/(1 + np.exp((t3-tmoh)))  + a4/(1 + np.exp((t4-tmoh)))  + a5/(1 + np.exp((t5-tmoh))) \
    + a6/(1 + np.exp((t6-tmoh))) + a7/(1 + np.exp((t7-tmoh))) 
    # + a8/(1 + np.exp((t8-tmoh)))

    beta_mean[:,0] = beta_mean[:,0] + beta_i[:,0]
    

    beta_e[:,0] = beta_i[:,0]

    idxmoh = 0

    #### With mobility tensor   

    FoI = np.zeros((len(tmoh),1))
        
    for kk in range(1,len(tmoh)):

        FoI[kk,0] = beta_e[kk-1,0] * (E[kk-1,0] + I[kk-1,0]) / N[kk-1,0]

        S[kk,0] = S[kk-1,0] + dt*(- FoI[kk,0] * S[kk-1,0])
        E[kk,0] = E[kk-1,0] + dt*(FoI[kk,0]*S[kk-1,0] - (gamma_i + gamma_e)*E[kk-1,0])
        
        I[kk,0] = I[kk-1,0] + dt*(gamma_i*E[kk-1,0] - (gamma_r + gamma_d)*I[kk-1,0])
        R[kk,0] = R[kk-1,0] + dt*(gamma_e*E[kk-1,0] + gamma_r*I[kk-1,0])
        D[kk,0] = D[kk-1,0] + dt*(gamma_d*I[kk-1,0])
        N[kk,0] = S[kk,0] +  E[kk,0] + I[kk,0] + R[kk,0]

 # Compute mean predictions
        S_mean[kk,0] = S_mean[kk,0] + S[kk,0]
        E_mean[kk,0] = E_mean[kk,0] + E[kk,0]
        I_mean[kk,0] = I_mean[kk,0] + I[kk,0]
        R_mean[kk,0] = R_mean[kk,0] + R[kk,0]
        D_mean[kk,0] = D_mean[kk,0] + D[kk,0]
        

        if( kk%ndiv == 0):
            idxmoh = int(kk/ndiv)

            if(idxmoh >= d_cut):

                I_pdf[idxmoh-d_cut,ns] = I[kk,0]



S_mean = S_mean/Nsamples
E_mean = E_mean/Nsamples
I_mean = I_mean/Nsamples
R_mean = R_mean/Nsamples
D_mean = D_mean/Nsamples

beta_mean = beta_mean/Nsamples


In [ ]:
E_mean[0,0] = sum(samples[8,:],1)/Nsamples
I_mean[0,0] = sum(samples[9,:],1)/Nsamples

S_mean[0,0] = S[0,0]
R_mean[0,0] = R[0,0]
D_mean[0,0] = D[0,0]

In [ ]:
d_cut = 160

pred_low = int(d_cut*ndiv)

pred_high = int(tlim*ndiv)

Nsamples = 500

for ns in range(Nsamples):


##### For Toronto
    a0 = samples[0,ns]
    a1 = samples[1,ns]
    t1 =  20
    a2 = samples[2,ns]
    t2 =  35
    a3 =  samples[3,ns]
    t3 = 60
    a4 =  samples[4,ns]
    t4 = 80
    a5 =  samples[5,ns]
    t5 =  140


    a6 =  samples[6,ns]
    t6 = 180
    a7 =  samples[7,ns]
    t7 =  190

    # Initial Conditions
    ####### CHANGE HERE ######################
    E[0,0] = samples[8,ns]
    I[0,0] = samples[9,ns]
    #####################################
    R[0,0] = 0
    D[0,0] = 0
    N[0,0] = total[0]
    S[0,0] = N[0,0] - E[0,0] - I[0,0] - R[0,0] - D[0,0]

    beta_i[:,0] = a0 + a1/(1 + np.exp((t1-tmoh))) +  a2/(1 + np.exp((t2-tmoh))) + a3/(1 + np.exp((t3-tmoh)))  + a4/(1 + np.exp((t4-tmoh))) + a5/(1 + np.exp((t5-tmoh))) \
    + a6/(1 + np.exp((t6-tmoh))) + a7/(1 + np.exp((t7-tmoh))) 
    # + a8/(1 + np.exp((t8-tmoh)))
    beta_e[:,0] = beta_i[:,0]

    if ns == 0:
        beta_append=beta_e
    else:
        beta_append=np.hstack((beta_append,beta_e))


    idxmoh = 0

    #### With mobility tensor   

    FoI = np.zeros((len(tmoh),1))
        
    for kk in range(1,len(tmoh)):

        FoI[kk,0] = beta_e[kk-1,0] * (E[kk-1,0] + I[kk-1,0]) / N[kk-1,0]

        S[kk,0] = S[kk-1,0] + dt*(- FoI[kk,0] * S[kk-1,0])
        E[kk,0] = E[kk-1,0] + dt*(FoI[kk,0]*S[kk-1,0] - (gamma_i + gamma_e)*E[kk-1,0])
        
        I[kk,0] = I[kk-1,0] + dt*(gamma_i*E[kk-1,0] - (gamma_r + gamma_d)*I[kk-1,0])
        R[kk,0] = R[kk-1,0] + dt*(gamma_e*E[kk-1,0] + gamma_r*I[kk-1,0])
        D[kk,0] = D[kk-1,0] + dt*(gamma_d*I[kk-1,0])
        N[kk,0] = S[kk,0] +  E[kk,0] + I[kk,0] + R[kk,0]


    if ns == 0:
        s_append=S[:,0]
        e_append=E[:,0]
        i_append=I[:,0]
        r_append=R[:,0]
        d_append=D[:,0]
    else:
        s_append=np.vstack((s_append,S[:,0]))
        e_append=np.vstack((e_append,E[:,0]))
        i_append=np.vstack((i_append,I[:,0]))
        r_append=np.vstack((r_append,R[:,0]))
        d_append=np.vstack((d_append,D[:,0]))


s_append=s_append.T
e_append=e_append.T
i_append=i_append.T
r_append=r_append.T
d_append=d_append.T

print(np.shape(i_append[pred_low:pred_high,:]))
print(np.shape(beta_append[pred_low:pred_high,:]))



def compute_bounds(arr):
    mean = arr.mean(axis=1)
    sd   = arr.std(axis=1)
    upper = mean + 1.96 * sd
    lower = mean - 1.96 * sd
    return mean, upper, lower

# For i_append
i_mean, i_upper, i_lower = compute_bounds(i_append)
np.savetxt("i_mean_MAP.csv", i_upper, delimiter=",")
np.savetxt("i_upper.csv", i_upper, delimiter=",")
np.savetxt("i_lower.csv", i_lower, delimiter=",")

# For e_append
e_mean, e_upper, e_lower = compute_bounds(e_append)
np.savetxt("e_mean_MAP.csv", e_upper, delimiter=",")
np.savetxt("e_upper.csv", e_upper, delimiter=",")
np.savetxt("e_lower.csv", e_lower, delimiter=",")

# For beta_append
b_mean, b_upper, b_lower = compute_bounds(beta_append)
np.savetxt("beta_mean_MAP.csv", b_upper, delimiter=",")
np.savetxt("beta_upper.csv", b_upper, delimiter=",")
np.savetxt("beta_lower.csv", b_lower, delimiter=",")


Igrd = np.linspace(0,20000,201)
Ipdf_0 = st.gaussian_kde(i_append[1650,:],bw_method = 0.2)
Ipdf_5 = st.gaussian_kde(i_append[1750,:],bw_method = 0.2)
Ipdf_10 = st.gaussian_kde(i_append[1850,:],bw_method = 0.2)
Ipdf_15 = st.gaussian_kde(i_append[1950,:],bw_method = 0.2)
# Ipdf_20 = st.gaussian_kde(i_append[2000,:],bw_method = 0.2)
Imax = np.max([Ipdf_0(Igrd),Ipdf_5(Igrd),Ipdf_10(Igrd),Ipdf_15(Igrd)])
# Imax = np.max([Ipdf_0(Igrd),Ipdf_5(Igrd),Ipdf_10(Igrd),Ipdf_15(Igrd),Ipdf_20(Igrd)])


Egrd = np.linspace(0,20000,201)
Epdf_0 = st.gaussian_kde(e_append[1650,:],bw_method = 0.2)
Epdf_5 = st.gaussian_kde(e_append[1750,:],bw_method = 0.2)
Epdf_10 = st.gaussian_kde(e_append[1850,:],bw_method = 0.2)
Epdf_15 = st.gaussian_kde(e_append[1950,:],bw_method = 0.2)
# Ipdf_20 = st.gaussian_kde(i_append[2000,:],bw_method = 0.2)
Emax = np.max([Epdf_0(Egrd),Epdf_5(Egrd),Epdf_10(Egrd),Epdf_15(Egrd)])
# Imax = np.max([Ipdf_0(Igrd),Ipdf_5(Igrd),Ipdf_10(Igrd),Ipdf_15(Igrd),Ipdf_20(Igrd)])



Dgrd = np.linspace(0,1000,201)
Dpdf_0 = st.gaussian_kde(d_append[1650,:],bw_method = 0.2)
Dpdf_5 = st.gaussian_kde(d_append[1750,:],bw_method = 0.2)
Dpdf_10 = st.gaussian_kde(d_append[1850,:],bw_method = 0.2)
Dpdf_15 = st.gaussian_kde(d_append[1950,:],bw_method = 0.2)
# Ipdf_20 = st.gaussian_kde(i_append[2000,:],bw_method = 0.2)
Dmax = np.max([Dpdf_0(Dgrd),Dpdf_5(Dgrd),Dpdf_10(Dgrd),Dpdf_15(Dgrd)])
# Imax = np.max([Ipdf_0(Igrd),Ipdf_5(Igrd),Ipdf_10(Igrd),Ipdf_15(Igrd),Ipdf_20(Igrd)])



Bgrd = np.linspace(0,0.3,201)
Bpdf_0 = st.gaussian_kde(beta_append[1650,:],bw_method = 0.2)
Bpdf_5 = st.gaussian_kde(beta_append[1750,:],bw_method = 0.2)
Bpdf_10 = st.gaussian_kde(beta_append[1850,:],bw_method = 0.2)
Bpdf_15 = st.gaussian_kde(beta_append[1950,:],bw_method = 0.2)
# Bpdf_20 = st.gaussian_kde(beta_append[2000,:],bw_method = 0.2)
Bmax = np.max([Bpdf_0(Igrd),Bpdf_5(Bgrd),Bpdf_10(Bgrd),Bpdf_15(Bgrd)])
# Bmax = np.max([Bpdf_0(Bgrd),Bpdf_5(Bgrd),Bpdf_10(Bgrd),Bpdf_15(Bgrd),Ipdf_20(Bgrd)])

t_data = np.arange(0,201)

f, ax = plt.subplots(1, figsize=(5,3))#15,3
plt.plot(tmoh[pred_low:pred_high], i_append[pred_low:pred_high,:], color=(142/255,186/255,217/255,0.5), linewidth=0.2, label='_nolegend_')
plt.plot(tmoh[pred_low:pred_high], all_truth[pred_low:pred_high,2], color='tab:red',linestyle='--', label='Estimated truth')

plt.plot(tmoh[pred_low:pred_high],i_mean[pred_low:pred_high], color='black',zorder=3, linestyle='--', label='mean of samples')
# Plot upper bound (red dashed)
plt.plot(tmoh[pred_low:pred_high],i_upper[pred_low:pred_high], color='red',linestyle='--', label='Upper 95% CI')
# Plot lower bound (red dashed)
plt.plot(tmoh[pred_low:pred_high],i_lower[pred_low:pred_high],color='red', linestyle='--', label='Lower 95% CI')

plt.scatter(t_data[160:201],x_data[160:201],color='w',edgecolors='k',zorder=3,alpha=0.4,label='Data')
plt.plot(tmoh[pred_low:pred_high], I_mean[pred_low:pred_high,0], color='tab:blue',linestyle='-', label='Mean')
plt.grid(True)
plt.plot(165+Ipdf_0(Igrd)*10/Imax,Igrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(175+Ipdf_5(Igrd)*10/Imax,Igrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(185+Ipdf_10(Igrd)*10/Imax,Igrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(195+Ipdf_15(Igrd)*10/Imax,Igrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
# plt.plot(200+Ipdf_20(Igrd)*10/Imax,Igrd*2794356,linewidth=1.5,c='k',linestyle=':',zorder=3)
# plt.xticks([272, 279, 286,293,300])
plt.xlim([160,200])
plt.ylim([0,1e4])
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel(r'Infectious compartment, $I(t)$', fontsize=12)
ax.legend(loc='upper left')
# ax.legend(loc='upper left', bbox_to_anchor=(0.5, -0.25), fancybox=True, shadow=True, ncol=3, fontsize=12)
plt.savefig(f'{figpath}/forecast_i_synth_noise10_initial_sigmoid.pdf',bbox_inches='tight')






f, ax = plt.subplots(1, figsize=(5,3))#15,3
plt.plot(tmoh[pred_low:pred_high], e_append[pred_low:pred_high,:], color=(142/255,186/255,217/255,0.5), linewidth=0.2, label='_nolegend_')
plt.plot(tmoh[pred_low:pred_high], all_truth[pred_low:pred_high,1], color='tab:red',linestyle='--', label='Truth')
plt.plot(tmoh[pred_low:pred_high], E_mean[pred_low:pred_high,0], color='tab:blue',linestyle='-', label='Mean')
plt.grid(True)
plt.plot(165+Epdf_0(Egrd)*10/Emax,Egrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(175+Epdf_5(Egrd)*10/Emax,Egrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(185+Epdf_10(Egrd)*10/Emax,Egrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(195+Epdf_15(Egrd)*10/Emax,Egrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
# plt.plot(200+Ipdf_20(Igrd)*10/Imax,Igrd*2794356,linewidth=1.5,c='k',linestyle=':',zorder=3)
# plt.xticks([272, 279, 286,293,300])
plt.xlim([160,200])
plt.ylim([0,1e4])
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel(r'Exposed compartment, $E(t)$', fontsize=12)
# ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), fancybox=True, shadow=True, ncol=3, fontsize=12)
plt.savefig(f'{figpath}/forecast_e_synth_noise10_initial_sigmoid.pdf',bbox_inches='tight')




f, ax = plt.subplots(1, figsize=(5,3))#15,3
plt.plot(tmoh[pred_low:pred_high], d_append[pred_low:pred_high,:], color=(142/255,186/255,217/255,0.5), linewidth=0.2, label='_nolegend_')
plt.plot(tmoh[pred_low:pred_high], all_truth[pred_low:pred_high,4], color='tab:red',linestyle='--', label='Truth')
plt.plot(tmoh[pred_low:pred_high], D_mean[pred_low:pred_high,0], color='tab:blue',linestyle='-', label='Mean')
plt.grid(True)
plt.plot(165+Dpdf_0(Dgrd)*10/Dmax,Dgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(175+Dpdf_5(Dgrd)*10/Dmax,Dgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(185+Dpdf_10(Dgrd)*10/Dmax,Dgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(195+Dpdf_15(Dgrd)*10/Dmax,Dgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
# plt.plot(200+Ipdf_20(Igrd)*10/Imax,Igrd*2794356,linewidth=1.5,c='k',linestyle=':',zorder=3)
# plt.xticks([272, 279, 286,293,300])
plt.xlim([160,200])
plt.ylim([0,1000])
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel(r'Deceased compartment, $D(t)$', fontsize=12)
# ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), fancybox=True, shadow=True, ncol=3, fontsize=12)
plt.savefig(f'{figpath}/forecast_d_synth_noise10_initial_sigmoid.pdf',bbox_inches='tight')





f, ax = plt.subplots(1, figsize=(5,3))#15,3
plt.plot(tmoh[pred_low:pred_high], beta_append[pred_low:pred_high,:], color=(142/255,186/255,217/255,0.5), linewidth=0.2, label='_nolegend_')
plt.plot(tmoh[pred_low:pred_high], beta_truth[pred_low:pred_high],linewidth=2,c='tab:red',linestyle='--', label='Truth')
plt.plot(tmoh[pred_low:pred_high], beta_mean[pred_low:pred_high,0], color='tab:blue',linestyle='-', label='Mean')
plt.plot(tmoh[pred_low:pred_high],b_upper[pred_low:pred_high], color='red',linestyle='--', label='Upper 95% CI')
# Plot lower bound (red dashed)
plt.plot(tmoh[pred_low:pred_high],b_lower[pred_low:pred_high],color='red', linestyle='--', label='Lower 95% CI')
plt.grid(True)
# plt.xticks([272, 279, 286,293,300])
plt.plot(165+Bpdf_0(Bgrd)*10/Bmax,Bgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(175+Bpdf_5(Bgrd)*10/Bmax,Bgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(185+Bpdf_10(Bgrd)*10/Bmax,Bgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.plot(195+Bpdf_15(Bgrd)*10/Bmax,Bgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
# plt.plot(200+Bpdf_20(Bgrd)*10/Bmax,Bgrd,linewidth=1.5,c='k',linestyle=':',zorder=3)
plt.xlim([160,200])
plt.ylim([0,0.3])
plt.xlabel('Time (days)', fontsize=12)
plt.ylabel(r'Infection rate parameter, $\beta(t)$', fontsize=12)
# ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), fancybox=True, shadow=True, ncol=3, fontsize=12)
plt.savefig(f'{figpath}/forecast_beta_synth_noise10_initial_sigmoid.pdf',bbox_inches='tight')
